In [1]:
#Trains the translator net
import random

import numpy as np
from Bio.Seq import Seq

import torch
from torch import nn
import torch.nn.functional as F

from utils import *
from optimus import *
from olg5utrdesign import *

device = torch.device('cuda:0')

In [ ]:
len_pro = 500
train_size = 10000
test_size = 1000

#Generate random DNA
def gen_random_dna(len_seqs, number_to_gen):
    #returns list of uniform randomly generated nucleotide strings of a given length
    return [''.join(random.choice(DNA_ALPHABET) for i in range(len_seqs)) for j in range(0, number_to_gen)]
all_dna  = gen_random_dna(len_pro * 3, train_size + test_size)
random.shuffle(all_dna)

#Generate protein sequences by translating DNA in all frames; forward strand
all_prot_f1 = [ str(Seq(s).translate()).replace("*", "X") for s in all_dna ]
all_prot_f2 = [ str(Seq(s[1:]).translate()).replace("*", "X") for s in all_dna ]
all_prot_f3 = [ str(Seq(s[2:]).translate()).replace("*", "X") for s in all_dna ]

#In reverse strand: reverse_complement, translate, then reverse
all_prot_r1 = [ str(Seq(s).reverse_complement().translate()).replace("*", "X")[::-1] for s in all_dna ]
all_prot_r2 = [ str(Seq(s).reverse_complement()[2:].translate()).replace("*", "X")[::-1] for s in all_dna ]
all_prot_r3 = [ str(Seq(s).reverse_complement()[1:].translate()).replace("*", "X")[::-1] for s in all_dna ]
all_prot = [all_prot_f1, all_prot_f2, all_prot_f3, all_prot_r1, all_prot_r2, all_prot_r3]

#seq to one hot tensor
all_dna = dna_to_onehot(all_dna, len(all_dna[0])).transpose(2, 1)
all_prot = [ aa_to_onehot(prot, len(prot[0])).transpose(2, 1) for prot in all_prot ]

#Split train-test sets
train_dna = all_dna[0:train_size]
test_dna = all_dna[train_size:len(all_dna)]
train_withstop = [ all_prot[i][0:train_size] for i in range(len(all_prot)) ]
test_withstop = [ all_prot[i][train_size:len(all_prot[i])] for i in range(len(all_prot)) ]

#torch.save([train_dna, test_dna, train_withstop, test_withstop], "translator_training_data.pth")
#train_dna, test_dna, train_withstop, test_withstop = torch.load("translator_training_data.pth", weights_only=False)

In [43]:
def criterion(pred, target, eps=1e-16):
    loss = torch.tensor([0.0]).to(device)
    for i in range(len(target)):
        loss += (-1.0 * torch.sum(target[i] * torch.log(pred[i] + eps), dim=1)).mean() #Cross entropy loss
    return loss

translator = Translator(1024).to(device)

#Training translation net
batch_size = 16
n_epoch = 2
n_train = len(train_withstop[0])

optimizer = torch.optim.AdamW(
    translator.parameters(),
    lr=0.001,
    betas=(0.9, 0.999),
    weight_decay=0.01
)

losses = []

for epoch in range(n_epoch):
    running_loss = 0.
    last_loss = 0.
    training_iter = 0
    last_index = 0
    print('epoch ' + str(epoch))
    while last_index < n_train:
        last_index_end = min(last_index + batch_size, n_train)
        input_onehot = train_dna[last_index:last_index_end]
        withstop = [ train_withstop[i][last_index:last_index_end, :, :].to(device) for i in range(len(train_withstop)) ]
        
        optimizer.zero_grad()
        out_withstop = translator(input_onehot.to(device), temperature=4.0)
        loss_withstop = criterion(out_withstop, withstop)
        loss = loss_withstop
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(translator.parameters(), max_norm=0.5)
        losses += [loss.detach().clone()]
        optimizer.step()
        
        running_loss += loss.item()
        last_index = last_index_end 
        training_iter += 1        
        
        if training_iter % 200 == 199:
            last_loss = running_loss / 200 # loss per batch
            print('  batch {} loss: {}'.format(training_iter + 1, last_loss))
            running_loss = 0.

epoch 0
  batch 200 loss: 3.4691380524635314
  batch 400 loss: 0.2932558208703995
  batch 600 loss: 0.7106260967254638
epoch 1
  batch 200 loss: 0.0
  batch 400 loss: 0.0
  batch 600 loss: 0.0


In [ ]:
#Check error rate on test set
with torch.inference_mode():
    pred_withstop = translator(test_dna.to(device), temperature=1.0)
    error_withstop = [ torch.stack([ torch.mean(torch.sum(torch.argmax(pred_withstop[i], 1) != torch.argmax(test_withstop[i].to(device), 1), dim=1)*1.0) / len_pro for i in range(len(pred_withstop)) ]) ]

print(error_withstop)

In [45]:
torch.save(translator.state_dict(), "./translator_cnn_1024ch.pth")